# img2txt: дерматоскопический пайплайн

**Пайплайн из 4 шагов, каждый принимает и возвращает DataFrame:**

1. **Извлечение признаков** — сегментация + 60+ признаков (цвет, форма, граница, текстура)
2. **Бакетирование** — числовые признаки -> категориальные метки
3. **Ранжирование** — нейросеть выбирает топ-10 важных признаков
4. **Генерация текста** — Mistral-7B генерирует клиническое описание на русском

---

# Часть 1. Обучение / исследование (batch)

In [ ]:
import pandas as pd
import torch

from extraction.feature_extraction_batch import extract_features_batch, images_to_df
from analysis.feature_bucketing_batch import bucket_features_batch, get_label_statistics
from importance.importance_inference import rank_features_batch
from generation.description_inference import generate_descriptions_batch
from generation.classification_types import ClassificationResult, Structure, FeatureType

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Пути (Kaggle)
IMAGE_DIR = "/kaggle/input/datasets/mihailodin1/all-image-skin"
YOLO_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_yolo.pt"
UNET_WEIGHTS = "/kaggle/input/weight-mask/weight/mask_builder_unet.pth"
IMPORTANCE_CHECKPOINT = "importance_checkpoints/best.pt"
FEATURES_CSV = "/kaggle/input/datasets/mihailodin1/features-img2txt/features_dataset.csv"

## Шаг 1. Извлечение признаков

In [ ]:
# Вариант A: извлечь признаки из директории с изображениями
df = images_to_df(IMAGE_DIR)
df = extract_features_batch(df, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS)
df.to_csv("features_dataset.csv", index=False)
print(f"Извлечено: {len(df)} изображений, успешно: {(df['status'] == 'success').sum()}")

In [ ]:
# Вариант B: загрузить ранее вычисленные признаки
df = pd.read_csv(FEATURES_CSV)
print(f"Загружено: {len(df)} изображений")

## Шаг 2. Бакетирование признаков

In [ ]:
df = bucket_features_batch(df)
print(f"Добавлены колонки: labels, features_organized, labels_json")

In [ ]:
# Статистика по меткам
stats = get_label_statistics(df)
if not stats.empty:
    print(stats.to_string())

In [ ]:
from importance.pseudo_importance import add_pseudo_important_labels

df = add_pseudo_important_labels(
    df,
    labels_col="labels",
    features_col="features_json",
    top_k=15,
    mode="combined",
    z_weight=1.5,
    prior_weight=0.8,
)
df["important_labels"] = df["pseudo_important_labels"]

print("Пример псевдо-меток:")
print(df["pseudo_important_labels"].iloc[0])

In [ ]:
from pathlib import Path

ANNOTATIONS_CSV = "annotations.csv"  # реальная разметка
# После фильтрации (ячейки ниже) замени на:
# ANNOTATIONS_CSV = "annotations_filtered.csv"

# Загружаем реальные аннотации
df_annot = pd.read_csv(ANNOTATIONS_CSV)

# Нормализуем пути: сравниваем только по имени файла (без директории)
# чтобы не зависеть от того, абсолютный путь или относительный
def _fname(p):
    return Path(str(p)).name

annot_map = {_fname(row["image_path"]): row["important_labels"]
             for _, row in df_annot.iterrows()}

# Подставляем реальные метки там, где есть аннотация,
# для остальных оставляем псевдо-метки из cell-9
def _pick_labels(row):
    key = _fname(row["image_path"])
    return annot_map[key] if key in annot_map else row["pseudo_important_labels"]

df["important_labels"] = df.apply(_pick_labels, axis=1)

n_real   = sum(1 for p in df["image_path"] if _fname(p) in annot_map)
n_pseudo = len(df) - n_real
print(f"Реальных аннотаций : {n_real}")
print(f"Псевдо-меток       : {n_pseudo}")
print(f"Итого              : {len(df)}")

# Сохраняем два CSV:
# 1. Только реальная разметка — используется как data_csv в train_importance
real_mask = df["image_path"].apply(_fname).isin(annot_map)
df_real_only = df[real_mask].copy()
df_real_only.to_csv("features_dataset_real.csv", index=False)

# 2. Только псевдо-разметка (без пересечений) — используется как pseudo_csv
df_pseudo_only = df[~real_mask].copy()
df_pseudo_only.to_csv("features_dataset_pseudo.csv", index=False)

print("\nСохранено:")
print(f"  features_dataset_real.csv   — {len(df_real_only)} строк (реальные)")
print(f"  features_dataset_pseudo.csv — {len(df_pseudo_only)} строк (псевдо, без пересечений)")

### 3b. Формирование датасета для обучения

In [ ]:
FEATURES_BUCKET_CSV = "features_dataset_bucket.csv"
ANNOTATIONS_CSV = "annotations.csv"

df_features = pd.read_csv(FEATURES_BUCKET_CSV)
df_annot = pd.read_csv(ANNOTATIONS_CSV)

df_train = df_features.merge(df_annot[["image_path", "important_labels"]], on="image_path")

print(f"Изображений с разметкой: {len(df_train)}")
df_train.to_csv("features_dataset_for_importance.csv", index=False)
print("Сохранено: features_dataset_for_importance.csv")

In [ ]:
from config.importance_config import FEAT_KEYS, FEAT_DIM, LABEL_NAMES, NUM_LABELS
import ast

df_check = pd.read_csv("features_dataset_for_importance.csv")

print(f"=== Датасет для обучения ===")
print(f"Строк: {len(df_check)}")
print(f"Колонок всего: {len(df_check.columns)}")

# ── Проверка признаков (FEAT_KEYS) ───────────────────────────────────────────
feat_present   = [k for k in FEAT_KEYS if k in df_check.columns]
feat_missing   = [k for k in FEAT_KEYS if k not in df_check.columns]
print(f"\n--- Признаки (FEAT_KEYS = {FEAT_DIM}) ---")
print(f"Найдено в колонках : {len(feat_present)}/{FEAT_DIM}")
if feat_missing:
    print(f"ОТСУТСТВУЮТ        : {feat_missing}")

# Доля непустых значений по каждому ключу
fill_rates = {k: df_check[k].notna().mean() for k in feat_present}
low_fill = {k: v for k, v in fill_rates.items() if v < 0.9}
if low_fill:
    print("Признаки с заполненностью < 90%:")
    for k, v in sorted(low_fill.items(), key=lambda x: x[1]):
        print(f"  {k:<35} {v*100:.1f}%")
else:
    print("Все признаки заполнены > 90% ✓")

# ── Проверка меток (important_labels) ────────────────────────────────────────
print(f"\n--- Метки (LABEL_NAMES = {NUM_LABELS}) ---")
labels_parsed = df_check["important_labels"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) else [])
)
label_keys_in_data = set(
    lbl.split(":")[0] for row in labels_parsed for lbl in row
)
unknown_keys = label_keys_in_data - set(LABEL_NAMES)
print(f"Уникальных ключей меток в датасете : {len(label_keys_in_data)}")
print(f"Из них в LABEL_NAMES               : {len(label_keys_in_data - unknown_keys)}")
if unknown_keys:
    print(f"НЕ в LABEL_NAMES (будут проигнорированы) : {sorted(unknown_keys)}")

lengths = labels_parsed.apply(len)
print(f"Меток на изображение: min={lengths.min()}  max={lengths.max()}  mean={lengths.mean():.1f}")

# ── Пример одной строки ───────────────────────────────────────────────────────
print(f"\n--- Пример строки [0] ---")
row0 = df_check.iloc[0]
print(f"image_path : {row0.get('image_path')}")
print(f"labels     : {labels_parsed.iloc[0]}")
print(f"features   : { {k: round(float(row0[k]), 3) for k in feat_present[:6] if pd.notna(row0[k])} } ...")

In [ ]:
# ── распределение значений по каждому признаку ───────────────────────────────
# Показываем только признаки, встречающиеся >= MIN_FREQ_PCT% изображений

MIN_FREQ_FOR_PLOT = 10  # % — порог для детального графика

val_counter = collections.defaultdict(collections.Counter)
for row in df_annot['important_labels']:
    for lbl in row:
        k, v = lbl.split(':', 1)
        val_counter[k][v] += 1

plot_features = [k for k, pct in zip(feat_names, feat_pcts) if pct >= MIN_FREQ_FOR_PLOT]
ncols = 3
nrows = (len(plot_features) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 3.2))
axes = axes.flatten()

for ax, feat in zip(axes, plot_features):
    vals = val_counter[feat]
    labels_v, counts_v = zip(*sorted(vals.items(), key=lambda x: -x[1]))
    colors = plt.cm.Set2(np.linspace(0, 1, len(labels_v)))
    ax.bar(range(len(labels_v)), counts_v, color=colors)
    ax.set_xticks(range(len(labels_v)))
    ax.set_xticklabels(labels_v, rotation=35, ha='right', fontsize=8)
    ax.set_title(f'{feat}\n(встреч: {feat_counter[feat]}, {feat_counter[feat]/N*100:.0f}%)', fontsize=9)
    ax.set_ylabel('кол-во', fontsize=8)

# скрываем пустые subplot-ы
for ax in axes[len(plot_features):]:
    ax.set_visible(False)

plt.suptitle(f'Значения признаков (≥{MIN_FREQ_FOR_PLOT}% изображений)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('feature_values_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: feature_values_distribution.png')

### Фильтрация лишних признаков

Задай порог частоты **`MIN_FREQ_PCT`** и/или явный список **`EXCLUDE_FEATURES`**.
Ячейка применит фильтр к аннотациям и сохранит `annotations_filtered.csv`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# НАСТРОЙКА ФИЛЬТРА
# ═══════════════════════════════════════════════════════════════════════════

# 1. Исключить признаки, встречающиеся меньше чем в MIN_FREQ_PCT% изображений
MIN_FREQ_PCT = 5.0  # поставь 0 чтобы отключить

# 2. Явно исключить конкретные признаки (добавляй сюда что считаешь лишним)
EXCLUDE_FEATURES = [
    # пример: 'lbp_entropy', 'color_balance_G', 'percent_red_pixels',
]

# ═══════════════════════════════════════════════════════════════════════════

# вычисляем итоговый список удаляемых признаков
rare = {k for k, v in feat_counter.items() if v / N * 100 < MIN_FREQ_PCT}
drop_set = rare | set(EXCLUDE_FEATURES)

print("Признаки, которые будут исключены:")
for f in sorted(drop_set):
    cnt = feat_counter.get(f, 0)
    print(f"  {f:<35} {cnt:>4} ({cnt/N*100:.1f}%)")

# применяем фильтр к каждой строке аннотаций
def _filter_labels(labels):
    return [lbl for lbl in labels if lbl.split(':')[0] not in drop_set]

df_filtered = df_annot.copy()
df_filtered['important_labels'] = df_filtered['important_labels'].apply(_filter_labels)

# статистика до / после
before = sum(len(r) for r in df_annot['important_labels'])
after  = sum(len(r) for r in df_filtered['important_labels'])
print(f"\nМеток до  фильтрации: {before}  (среднее на изображение: {before/N:.1f})")
print(f"Меток после фильтрации: {after}   (среднее на изображение: {after/N:.1f})")
print(f"Исключено признаков: {len(drop_set)} из {len(feat_counter)}")

# сохраняем
ANNOTATIONS_FILTERED = 'annotations_filtered.csv'
df_filtered['important_labels'] = df_filtered['important_labels'].apply(str)
df_filtered.to_csv(ANNOTATIONS_FILTERED, index=False)
print(f"\nСохранено: {ANNOTATIONS_FILTERED} ({len(df_filtered)} строк)")

# ── подсказка для следующего шага ───────────────────────────────────────────
print("\n" + "="*60)
print("Следующий шаг:")
print(f"  Замени ANNOTATIONS_CSV = '{ANNOTATIONS_FILTERED}'")
print("  в ячейке 3b и перезапусти её, чтобы обучить на")
print("  отфильтрованных метках.")
print("="*60)


In [ ]:
# ── сравнение до/после фильтрации (side-by-side бар-чарт) ────────────────────
df_filtered_r = pd.read_csv(ANNOTATIONS_FILTERED)
df_filtered_r['important_labels'] = df_filtered_r['important_labels'].apply(ast.literal_eval)

feat_after = collections.Counter(
    lbl.split(':')[0]
    for row in df_filtered_r['important_labels']
    for lbl in row
)
feat_kept = sorted(feat_after.items(), key=lambda x: -x[1])

fig, ax = plt.subplots(figsize=(10, max(5, len(feat_kept) * 0.35)))
y2 = np.arange(len(feat_kept))
ax.barh(y2, [v / N * 100 for _, v in feat_kept], color='steelblue', height=0.7)
ax.set_yticks(y2)
ax.set_yticklabels([k for k, _ in feat_kept], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('% изображений')
ax.set_title(f'Признаки после фильтрации (осталось {len(feat_kept)} из {len(feat_counter)})', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
for bar, (_, cnt) in zip(ax.patches, feat_kept):
    pct = cnt / N * 100
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.savefig('feature_distribution_filtered.png', dpi=150, bbox_inches='tight')
plt.show()
print('График сохранён: feature_distribution_filtered.png')

### Диагностика утечки RAM (tracemalloc)

Запускает фоновый поток, который каждые 30 секунд печатает топ-15 строк, аллоцирующих больше всего памяти, и diff относительно базового снимка. Запусти эту ячейку **до** ячейки обучения.

In [ ]:
import tracemalloc, threading, time, gc, os

# Останавливаем предыдущий мониторинг, если был
_TM_STOP = globals().get("_TM_STOP")
if _TM_STOP is not None:
    _TM_STOP.set()
    time.sleep(0.1)

tracemalloc.stop()
tracemalloc.start(25)  # 25 кадров стека на аллокацию
gc.collect()
_baseline = tracemalloc.take_snapshot()
_TM_STOP = threading.Event()

def _proc_rss_mb():
    try:
        with open(f"/proc/{os.getpid()}/status") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    return int(line.split()[1]) / 1024
    except Exception:
        return float("nan")

def _monitor(interval=30, top_n=15):
    step = 0
    while not _TM_STOP.wait(interval):
        step += 1
        gc.collect()
        snap = tracemalloc.take_snapshot()
        diff = snap.compare_to(_baseline, "lineno")
        rss = _proc_rss_mb()
        print(f"\n=== tracemalloc tick #{step}  RSS={rss:.0f} MB ===")
        for stat in diff[:top_n]:
            frame = stat.traceback[0]
            print(f"  +{stat.size_diff/1e6:+7.1f} MB  size={stat.size/1e6:6.1f} MB  count_diff={stat.count_diff:+d}")
            print(f"      {frame.filename}:{frame.lineno}")
        # топ-3 покажем с полным стеком — там видно, кто реально вызывает аллокацию
        print("  -- top-3 traceback:")
        for stat in diff[:3]:
            print(f"  +{stat.size_diff/1e6:+.1f} MB at:")
            for line in stat.traceback.format()[-6:]:
                print("     " + line)

_thread = threading.Thread(target=_monitor, daemon=True)
_thread.start()
print(f"tracemalloc started, baseline RSS={_proc_rss_mb():.0f} MB. Run training cell now.")
print("Чтобы остановить мониторинг: _TM_STOP.set()")


In [ ]:
from importance.train_importance import train_importance

best_score = train_importance(
    data_csv="features_dataset_real.csv",
    image_dir=IMAGE_DIR,
    pseudo_csv="features_dataset_pseudo.csv",
    pseudo_weight=0.3,
    backbone="efficientnet_b0",
    epochs=50,
    batch_size=64,          # 32/GPU × 2 GPU
    lr=1e-4,
    freeze_backbone_epochs=5,
    label_smoothing=0.05,
    num_workers=4,          # параллельная загрузка данных
    use_amp=True,           # mixed precision на T4
    multi_gpu=True,         # использовать обе GPU через DataParallel
    out_dir="importance_checkpoints",
)
print(f"Best validation score: {best_score:.4f}")

### 3c. Ранжирование обученной моделью (batch)

In [ ]:
from importance.train_importance import train_importance

best_score = train_importance(
    data_csv="features_dataset_for_importance.csv",
    image_dir=IMAGE_DIR,
    backbone="efficientnet_b0",
    epochs=50,
    batch_size=64,
    lr=1e-4,
    freeze_backbone_epochs=5,
    label_smoothing=0.05,
    num_workers=4,
    use_amp=True,
    multi_gpu=True,
    out_dir="importance_checkpoints",
)
print(f"Best validation score: {best_score:.4f}")

## Шаг 4. Генерация клинических описаний (Mistral-7B)

Первый вызов загружает модель (~14GB). Последующие используют кэш.

In [ ]:
df = generate_descriptions_batch(df, device=device)

print("Пример описания:")
print(df["description"].iloc[0])

In [ ]:
# Сохранение результатов
output_csv = "features_with_descriptions.csv"
df.to_csv(output_csv, index=False)
print(f"Сохранено: {len(df)} строк в {output_csv}")

---

# Часть 2. Инференс (single image)

Тот же пайплайн, но для одного изображения — однострочный DataFrame.

In [ ]:
image_path = "/path/to/lesion.jpg"

# Шаг 1: извлечение признаков
df_single = pd.DataFrame([{"image_path": image_path}])
df_single = extract_features_batch(df_single, yolo_weights=YOLO_WEIGHTS, unet_weights=UNET_WEIGHTS, verbose=False)

# Шаг 2: бакетирование
df_single = bucket_features_batch(df_single, verbose=False)

# Шаг 3: ранжирование
df_single = rank_features_batch(df_single, importance_model_path=IMPORTANCE_CHECKPOINT, device=device, verbose=False)

# Шаг 4: генерация описания (с классификацией)
df_single["classification"] = ClassificationResult(
    feature_type=FeatureType.SINGLE,
    structure=Structure.GLOBULES,
    properties=["однородный"],
    final_class="Меланома",
)
df_single = generate_descriptions_batch(df_single, classification_col="classification", device=device, verbose=False)

print(df_single.iloc[0]["description"])